In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import joblib
import os

df = pd.read_csv("data/documents_dataset.csv")
print(df.shape)
print(df["label"].value_counts())
df.head()

(222, 2)
label
declaration_sinistre    35
constat_amiable         32
rapport_expertise       32
permis_conduire         31
devis                   31
facture                 31
carte_grise             30
Name: count, dtype: int64


,text,label
0,republique tunisienne ministere de interieur p...,permis_conduire
1,auto pieces sfax amenagement renovation devis ...,devis
2,constat amiable accident automobile signer obl...,constat_amiable
3,maison deco amenagement renovation devis 2025 ...,devis
4,republique tunisienne ministere de interieur p...,permis_conduire


In [11]:
X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 177, Test: 45


In [12]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),   # unigrams + bigrams — captures phrases like "carte grise"
    min_df=2,              # ignore terms that appear in fewer than 2 documents
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(X_train_vec.shape)

(177, 2205)


In [13]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_vec, y_train)

y_pred_lr = log_reg.predict(X_test_vec)
print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr))

=== Logistic Regression ===
                      precision    recall  f1-score   support

         carte_grise       1.00      1.00      1.00         6
     constat_amiable       1.00      1.00      1.00         7
declaration_sinistre       1.00      1.00      1.00         7
               devis       1.00      1.00      1.00         6
             facture       1.00      1.00      1.00         6
     permis_conduire       1.00      1.00      1.00         6
   rapport_expertise       1.00      1.00      1.00         7

            accuracy                           1.00        45
           macro avg       1.00      1.00      1.00        45
        weighted avg       1.00      1.00      1.00        45



In [14]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_vec, y_train)

y_pred_rf = rf.predict(X_test_vec)
print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))

=== Random Forest ===
                      precision    recall  f1-score   support

         carte_grise       0.86      1.00      0.92         6
     constat_amiable       1.00      1.00      1.00         7
declaration_sinistre       1.00      1.00      1.00         7
               devis       1.00      1.00      1.00         6
             facture       1.00      1.00      1.00         6
     permis_conduire       1.00      0.83      0.91         6
   rapport_expertise       1.00      1.00      1.00         7

            accuracy                           0.98        45
           macro avg       0.98      0.98      0.98        45
        weighted avg       0.98      0.98      0.98        45



In [15]:
# Cell 6 — Cross-validation (separate vectorizer, doesn't affect the original)
cv_vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2)
X_full_vec = cv_vectorizer.fit_transform(X)

cv_scores_lr = cross_val_score(LogisticRegression(max_iter=1000, random_state=42), X_full_vec, y, cv=5, scoring="f1_macro")
cv_scores_rf = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=42), X_full_vec, y, cv=5, scoring="f1_macro")

print(f"Logistic Regression — CV F1-macro: {cv_scores_lr.mean():.3f} (+/- {cv_scores_lr.std():.3f})")
print(f"Random Forest       — CV F1-macro: {cv_scores_rf.mean():.3f} (+/- {cv_scores_rf.std():.3f})")

Logistic Regression — CV F1-macro: 0.996 (+/- 0.009)
Random Forest       — CV F1-macro: 0.991 (+/- 0.011)


In [16]:
from generate_dataset import clean_text
holdout_samples = [
    ("""Formulaire de declaration de sinistre automobile. Je soussigne Mounir Ayari
    domicilie a Sfax declare avoir subi un accident de la circulation survenu le
    03/09/2025 vers 09h15 au niveau du rond point Complexe Sportif. Mon vehicule
    a percute un poteau suite a un freinage brusque. Aucun blesse n est a deplorer.
    Police d assurance numero 55214499 souscrite aupres de Comar. Un constat
    amiable a ete etabli sur les lieux.""", "declaration_sinistre"),

    ("""Objet accident survenu. Assure Monsieur Bilel Chaabane. Sinistre du type
    collision arriere survenu le 21 fevrier 2026 sur la route de Tunis
    Gouvernorat Ariana. Contrat n 990211. Compagnie GAT Assurances. Dommages
    materiels constates sur le pare choc et le coffre. Autorites non prevenues.""", "declaration_sinistre"),

    ("""Ministere des Transports Republique Tunisienne. Document officiel
    attestant l immatriculation du vehicule suivant plaque 88 tunis 2201 mise
    en circulation initiale le 12 fevrier 2019. Constructeur Toyota modele
    Yaris carburant essence 6 chevaux fiscaux capacite 5 places categorie
    tourisme. Chassis JTDBT923000456712 titulaire Sami Fradi domicilie a Sousse.""", "carte_grise"),

    ("""Certificat officiel d immatriculation delivre par le ministere du
    transport. Vehicule Dacia Duster plaque 77 tunis 9034. Premiere mise en
    circulation 2021. 8 chevaux fiscaux 5 places usage particulier. Numero
    de chassis VF7XXXXX. Adresse du proprietaire Nabeul.""", "carte_grise"),

    ("""Republique Tunisienne Ministere de l Interieur. Le present permis
    autorise Monsieur Fares Zouari ne le 14 aout 1990 a Monastir a conduire
    les vehicules de categorie B. Delivre le 02 janvier 2015 a Monastir
    valable jusqu au 02 janvier 2035. Numero de permis 884213.""", "permis_conduire"),

    ("""Document officiel. Titulaire Mme Salma Belhadj domiciliee a Bizerte
    categorie autorisee B et A1. Date de naissance 5 mai 1988 lieu de
    naissance Bizerte. Delivrance le 19 juin 2010 par les autorites de
    Bizerte.""", "permis_conduire"),

    ("""Facture emise par la societe Tunis Informatique sise a Ariana MF
    2233445 A M 000. Numero FT 2026 0091 date 14 janvier 2026. Client Rania
    Mabrouk adresse Ariana. Articles vendus un ordinateur portable et une
    souris. Montant hors taxe 2450 TVA 19 pourcent total ttc 2915 500.
    Merci de votre confiance.""", "facture"),

    ("""Facture N FA 552 2025. Emise le 8 octobre 2025 par Boutique Elegance
    Sfax. Client Hela Trabelsi. Designation vetements et accessoires
    quantite 3 prix unitaire 85 montant 255 total a payer 303 450 dinars
    TVA incluse.""", "facture"),

    ("""Devis etabli par l entreprise Renovation Plus specialisee en travaux
    de plomberie. Destinataire M Adel Ghanmi adresse La Marsa. Prestations
    prevues remplacement de la tuyauterie et installation d un chauffe eau.
    Cout estime hors taxe 1800 dinars TVA 19 pourcent total ttc 2142 devis
    valable un mois a compter de la date d emission.""", "devis"),

    ("""Proposition commerciale N DV 2026 044 emise par Peinture Pro. Client
    Nizar Selmi. Travaux prevus peinture facade exterieure surface 120
    metres carres. Montant hors taxe 3200 tva 608 total ttc 3808. Validite
    de l offre 45 jours.""", "devis"),

    ("""Formulaire de constat amiable rempli suite a une collision survenue
    avenue de la republique sfax le 19 11 2025 a 17h40. Vehicule A conduit
    par Mehdi Aouadi assure aupres de Comar police 447712 marque Kia Picanto
    immatricule 33 tunis 1290 dommages sur l aile avant droite. Vehicule B
    conduit par Emna Riahi assuree aupres de Star police 118820 marque Seat
    Ibiza immatricule 12 tunis 8890 dommages sur le pare choc arriere.
    Croquis joint. Signatures des deux conducteurs presentes.""", "constat_amiable"),

    ("""Accident de la circulation constate a l amiable entre deux
    automobilistes sur la route nationale numero 1 pres de Msaken le 2 aout
    2025. Premier vehicule Peugeot 208 conducteur Yosra Kammoun police d
    assurance 774411 chez Lloyd. Second vehicule Renault Symbol conducteur
    Anis Baccouche police 990022 chez Maghrebia. Degats constates portiere
    et retroviseur casse.""", "constat_amiable"),

    ("""Rapport etabli par le cabinet Auto Expert Tunisie suite a la mission
    d expertise numero EXP 2026 112 concernant le sinistre declare par Mme
    Ines Karray contrat 445210. L expert s est deplace sur les lieux le 20
    mars 2026 afin d evaluer les dommages consecutifs a une collision
    laterale. Les degats releves concernent la portiere avant gauche et le
    retroviseur estimation totale 1950 dinars. Rapport signe par l expert
    Nabil Rekik.""", "rapport_expertise"),

    ("""Cabinet d expertise Auto Diagnostic. Mission N 2025 887 relative au
    sinistre de M Karim Jaziri police 112233. Accident survenu le 11
    decembre 2025 choc arriere sur autoroute A1. Evaluation des reparations
    necessaires pare choc arriere plus feu arriere droit montant estime
    1275 dinars. Conclusion dommages compatibles avec les circonstances
    declarees.""", "rapport_expertise"),
]

correct = 0
print(f"{'True':<22} {'Predicted':<22} {'Conf':<6} Match")
print("-" * 60)
for raw_text, true_label in holdout_samples:
    cleaned = clean_text(raw_text)
    vec = vectorizer.transform([cleaned])
    pred = log_reg.predict(vec)[0]
    confidence = log_reg.predict_proba(vec).max()
    match = pred == true_label
    correct += match
    print(f"{true_label:<22} {pred:<22} {confidence:.2f}   {'✓' if match else '✗'}")

print(f"\nHoldout accuracy: {correct}/{len(holdout_samples)} = {correct/len(holdout_samples):.1%}")

True                   Predicted              Conf   Match
------------------------------------------------------------
declaration_sinistre   declaration_sinistre   0.30   ✓
declaration_sinistre   declaration_sinistre   0.21   ✓
carte_grise            carte_grise            0.37   ✓
carte_grise            carte_grise            0.59   ✓
permis_conduire        permis_conduire        0.67   ✓
permis_conduire        permis_conduire        0.18   ✓
facture                facture                0.37   ✓
facture                facture                0.25   ✓
devis                  devis                  0.28   ✓
devis                  devis                  0.39   ✓
constat_amiable        constat_amiable        0.31   ✓
constat_amiable        constat_amiable        0.23   ✓
rapport_expertise      rapport_expertise      0.29   ✓
rapport_expertise      rapport_expertise      0.28   ✓

Holdout accuracy: 14/14 = 100.0%


In [17]:
final_vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2)
X_all_vec = final_vectorizer.fit_transform(X)

final_model = LogisticRegression(max_iter=1000, random_state=42)
final_model.fit(X_all_vec, y)

os.makedirs("models", exist_ok=True)
joblib.dump(final_model, "models/classifier.joblib")
joblib.dump(final_vectorizer, "models/vectorizer.joblib")

print("Saved model and vectorizer to ml/models/")

Saved model and vectorizer to ml/models/


In [18]:
# Reload from disk exactly as the API will, and re-verify the holdout set
loaded_model = joblib.load("models/classifier.joblib")
loaded_vectorizer = joblib.load("models/vectorizer.joblib")

sample_text = clean_text(holdout_samples[0][0])
vec = loaded_vectorizer.transform([sample_text])
print(loaded_model.predict(vec), loaded_model.predict_proba(vec).max())

['declaration_sinistre'] 0.3169846212235263
